In [15]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import KFold

import napari
import tifffile as tiff

import topometrics.leaderboard
import matplotlib.pyplot as plt

import numpy as np
from tqdm import tqdm

import cc3d
import numpy as np
from cucim.skimage.measure import euler_number, label
from skimage.morphology import binary_dilation, skeletonize
import cupy as cp
import skimage.measure as ski

from pathlib import Path

In [16]:
# COMP METRICS PARAMS
surface_tolerance: float = 4.0
voi_connectivity: int = 26
voi_transform: str = 'one_over_one_plus'
voi_alpha: float = 0.3
topo_weight: float = 0.3
surface_dice_weight: float = 0.35
voi_weight: float = 0.35


In [17]:
def calc_score(labelarr, predarr):
    score_report = topometrics.leaderboard.compute_leaderboard_score(
        predictions=predarr,
        labels=labelarr,
        dims=(0, 1, 2),
        spacing=(1.0, 1.0, 1.0),  # (z, y, x)
        surface_tolerance=surface_tolerance,  # in spacing units
        voi_connectivity=voi_connectivity,
        voi_transform=voi_transform,
        voi_alpha=voi_alpha,
        combine_weights=(topo_weight, surface_dice_weight, voi_weight),  # (Topo, SurfaceDice, VOI)
        fg_threshold=None,  # None => legacy "!= 0"; else uses "x > threshold"
        ignore_label=2,  # voxels with this GT label are ignored
        ignore_mask=None,  # or pass an explicit boolean mask
    )

    return score_report
    

def calculate_betti_numbers(volume):
    """
    Calculates the first three Betti numbers (b0, b1, b2) of a 3D binary volume.
    
    Parameters:
    volume (np.ndarray): 3D binary array (0 for background, 1 for object).
    
    Returns:
    dict: {'b0': int, 'b1': int, 'b2': int}
    """
    # 1. Pad volume to ensure "background" is continuous around the object
    # This prevents voids from "leaking" out of the image bounds
    padded_vol = np.pad(volume, 1, mode='constant', constant_values=0)
    
    # 2. Calculate Beta 0 (Connected Components)
    # We use connectivity=3 for 26-neighbor connectivity (faces + edges + corners)
    labeled_array, b0 = label(padded_vol, connectivity=3, return_num=True)
    
    # 3. Calculate Beta 2 (Voids/Cavities)
    # We invert the image to look at the background
    inverse_vol = 1 - padded_vol
    
    # We use connectivity=1 for 6-neighbor connectivity (faces only)
    # This is the topological dual of the 26-connectivity used for foreground
    _, num_background_components = label(inverse_vol, connectivity=1, return_num=True)
    
    # The background count includes the "infinite" exterior surrounding the object.
    # We subtract 1 to count only the internal voids.
    b2 = num_background_components - 1
    
    # 4. Calculate Euler Characteristic (Chi)
    # skimage implementation uses connectivity=3 (26-connectivity) by default for 3D
    chi = euler_number(padded_vol, connectivity=3)
    
    # 5. Solve for Beta 1 (Tunnels/Handles) using Euler-Poincare formula
    # Chi = b0 - b1 + b2  =>  b1 = b0 + b2 - Chi
    b1 = b0 + b2 - chi
    
    return b0,b1,b2


def get_tunnel_skeleton(volume):
    """
    Returns a skeletonized version of the volume. 
    Loops in this skeleton represent the Beta 1 tunnels.
    """
    # 1. Get voids to fill them in
    voids = get_voids_mask(volume)
    
    # 2. Fill the voids! 
    # This is crucial. A hollow sphere has a void but no tunnel. 
    # If we skeletonize a hollow sphere without filling, we get a surface, not a line.
    # By filling it, we ensure we only look at genuine handles.
    filled_volume = volume | voids
    
    # 3. Skeletonize the filled volume
    # This shrinks the object to its topological core (1-pixel wide lines)
    skeleton = skeletonize(filled_volume)
    
    return skeleton

def get_voids_mask(volume):
    """
    Returns a binary mask where 1 = void (cavity) inside the object.
    """
    # 1. Pad to ensure the "true" background is connected around the object
    padded = np.pad(volume, 1, constant_values=0)
    
    # 2. Invert: Look at the background
    background = 1 - padded
    
    # 3. Label background components (using 6-connectivity for strict encapsulation)
    bg_labels, num_bg = ski.label(background, connectivity=1, return_num=True)
    
    # 4. Identify the "exterior" component (it touches the corner)
    exterior_label = bg_labels[0, 0, 0]
    
    # 5. Create mask of ONLY the internal voids (all bg labels except exterior)
    # The mask is True where label is not 0 (foreground) and not exterior
    voids_mask_padded = (bg_labels != exterior_label) & (bg_labels != 0)
    
    # 6. Unpad to return to original size
    return voids_mask_padded[1:-1, 1:-1, 1:-1]


In [11]:
ids = [x.stem for x in Path("../../data/train_images/").glob("*.tif")]

In [12]:
betti_numbers = np.zeros(3)

for id in tqdm(ids):
    
    lbl = tiff.imread(f"../../data/train_labels/{id}.tif")
    ignoremask = lbl == 2
    lbl[ignoremask] = 0

    cclbls = cc3d.connected_components(lbl, connectivity=26)

    lbl = cp.asarray(lbl)
    result = np.array(calculate_betti_numbers(lbl))
    betti_numbers += np.array(result)

    if np.max(cclbls) != result[0] or result[1] >= 1 or result[2] >= 1 or np.any(result < 0):
        print(id,np.max(cclbls), result)
    

  1%|█                                                                                                                                                                             | 5/786 [00:00<01:50,  7.06it/s]

2768504081 5 [5 1 0]
1578248244 14 [14  2  0]


  2%|██▋                                                                                                                                                                          | 12/786 [00:01<01:24,  9.18it/s]

108672114 13 [13  6  0]


  2%|███▌                                                                                                                                                                         | 16/786 [00:01<01:20,  9.58it/s]

...

 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 765/786 [01:21<00:02,  9.34it/s]

1440669255 6 [ 6  3 12]


 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 781/786 [01:23<00:00,  9.05it/s]

315189226 13 [13  1  0]
3248922039 16 [16  0  1]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 786/786 [01:23<00:00,  9.38it/s]


In [46]:
id = "3919317307"
vol = tiff.imread(f"../../data/train_images/{id}.tif")  # shape: (Z, Y, X)
lbl = tiff.imread(f"../../data/train_labels/{id}.tif")  # shape: (Z, Y, X)

ignoremask = lbl == 2
lbl[ignoremask] = 0

cclbls = cc3d.connected_components(lbl, connectivity=26)

cc = np.max(cclbls)

lbl = cp.asarray(lbl)
result = calculate_betti_numbers(lbl)
print(f"ID: {id}, Connected Components: {cc}, Betti Numbers: {result}")

viewer = napari.Viewer()

layers = []

for i in range(1, np.max(cclbls) + 1):
    layer = cclbls.copy()
    layermask = layer != i
    layer[layermask] = 0
    viewer.add_labels(layer, name=f"layer_{i}")
    layers.append(layer)
    layer = cp.asarray(layer)
    print(f"{i=}, betti_numbers: {calculate_betti_numbers(layer)}")
    
napari.run()

ID: 3919317307, Connected Components: 4, Betti Numbers: (4, 12, 11)
i=1, betti_numbers: (1, 10, 11)
i=2, betti_numbers: (1, 1, 1)
i=3, betti_numbers: (1, 3, 1)
i=4, betti_numbers: (1, 1, 1)


In [49]:
import matplotlib.pyplot as plt

def visualize_topology(volume):
    fg = volume != 0
    volume[fg] = 1

    # Invert the volume and then perform cca, because beta2 (void) will be surrounded by fg
    inv = cc3d.connected_components(1- volume, connectivity=26)
    print(inv.max())
    x = cc3d.statistics(inv)
    print(x)


# # Run visualization
visualize_topology(layers[1])

1
{'voxel_counts': array([  311681, 32456319], dtype=uint32), 'bounding_boxes': [(slice(3, 317, None), slice(3, 317, None), slice(103, 271, None)), (slice(0, 320, None), slice(0, 320, None), slice(0, 320, None))], 'centroids': array([[159.16986919, 158.294949  , 188.25329103],
       [159.50317028, 159.51157221, 159.22387952]])}


In [39]:
np.unique(layers[11])

array([0, 1], dtype=uint32)

In [51]:
lbl = cp.asnumpy(lbl)

In [52]:
calc_score(lbl,lbl)

LeaderboardReport(score=0.9999999999999999, topo=TopoReport(toposcore=1.0, topoF1_by_dim={0: 1.0, 1: 1.0, 2: nan}, counts_by_dim={0: (18, 18, 18), 1: (70, 70, 70), 2: (0, 0, 0)}, dims=[0, 1, 2]), surface_dice=1.0, voi=VOIReport(voi_total=1.1743769379138622e-16, voi_split=5.871884689569311e-17, voi_merge=5.871884689569311e-17, voi_score=1.0, n_foreground=1001666, connectivity=26), params={'dims': [0, 1, 2], 'topo_weights': None, 'fg_threshold': None, 'surface_tolerance': 4.0, 'spacing': (1.0, 1.0, 1.0), 'voi_connectivity': 26, 'voi_transform': 'one_over_one_plus', 'voi_alpha': 0.3, 'combine_weights': (0.3, 0.35, 0.35), 'ignore_label': 2, 'ignored_voxels': 0})